In [1]:
pip install lightgbm scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 2 — Load Libraries

import pandas as pd
import numpy as np

from lightgbm import LGBMRanker

from sklearn.model_selection import train_test_split

In [3]:
# Step 3 — Read CSV training data 

train_df = pd.read_csv(
    "data/processed/training_data.csv"
)
print(train_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/training_data.csv'

In [11]:
# Step 4 — Select Ranking Features

feature_columns = [
    "bm25_score",
    "rating",
    "review_count",
    "price",
    "is_best_seller",
    "bought_last_month"
]

In [13]:
# Step 5 — Create Feature Matrix

X = train_df[feature_columns]
y = train_df["relevance"]

In [15]:
# Step 6 — Create Query Groups
group = (
    train_df
    .groupby("query")
    .size()
    .to_numpy()
)

print(group)


[20 20 20 20 20]


In [16]:
# Step 6 — Train LambdaMART Model

ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=100
)

ranker.fit(
    X,
    y,
    group=group
)

print("LambdaMART model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 46
[LightGBM] [Info] Number of data points in the train set: 100, number of used features: 4
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

# Predict Ranking Scores

In [17]:
predictions = ranker.predict(X)
print(predictions[:10])

[-8.18576412 -8.19658479 -8.18576412 -8.18576412 -8.18576412 -8.18576412
 -8.18576412 -8.18576412 -7.88432214 -8.18576412]


In [18]:
# Step 2 — Attach Predictions To DataFrame

train_df["predicted_score"] = predictions
print(train_df.head())

  query  product_index  relevance  bm25_score  rating  review_count  price  \
0   कैप           2084          3    4.233124     0.0             0    0.0   
1   कैप           2794          3    4.218292     0.0             0  475.0   
2   कैप           2125          3    4.201487     0.0             0    0.0   
3   कैप           2118          3    4.201487     0.0             0    0.0   
4   कैप           2577          3    4.149776     3.4             3  279.0   

   is_best_seller  bought_last_month  predicted_score  
0               0                  0        -8.185764  
1               0                  0        -8.196585  
2               0                  0        -8.185764  
3               0                  0        -8.185764  
4               0                  0        -8.185764  


In [19]:
# Step 3 — View AI-Ranked Results

ranked_results = train_df.sort_values(
    by="predicted_score",
    ascending=False
)

print(
    ranked_results[
        [
            "query",
            "bm25_score",
            "rating",
            "review_count",
            "predicted_score"
        ]
    ].head(10)
)

   query  bm25_score  rating  review_count  predicted_score
77   बैग    5.389049     4.0           205         3.994317
63   बैग    7.287436     4.1           290         3.802840
61   बैग    8.256994     4.4           115         2.961169
74   बैग    5.484293     4.1           108         2.912620
62   बैग    8.042978     3.7           738        -1.717537
60   बैग    8.482710     3.7           198        -2.029768
78   बैग    5.208151     4.8            93        -2.760756
64   बैग    6.960508     3.7           107        -2.865169
73   बैग    5.582965     5.0             1        -2.967388
28  जूते    0.000000     5.0             1        -3.072976


# Mini Pipeline Architecture

In [24]:
# Step 1 — Retrieval Pipeline

def retrieve_candidates(query, top_k=20):

    print("\n[Retrieval Pipeline Started]\n")

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_n = np.argsort(scores)[::-1][:top_k]

    print(f"Query: {query}")

    print(f"\nTop {top_k} candidates retrieved!")

    return top_n

In [25]:
# Step 2 — Feature Pipeline

def build_feature_matrix(query, candidate_indices):

    print("\n[Feature Engineering Pipeline Started]\n")

    feature_rows = []

    for idx in candidate_indices:

        features = extract_features(
            query,
            idx
        )

        feature_rows.append([
            features["bm25_score"],
            features["rating"],
            features["review_count"],
            features["price"],
            features["is_best_seller"],
            features["bought_last_month"]
        ])

    print("Feature matrix created successfully!")

    return np.array(feature_rows)

In [26]:
# Step 3 — Ranking Pipeline

def rank_products(feature_matrix):

    print("\n[AI Ranking Pipeline Started]\n")

    predicted_scores = ranker.predict(
        feature_matrix
    )

    ranked_indices = np.argsort(
        predicted_scores
    )[::-1]

    print("Products ranked successfully!")

    return ranked_indices, predicted_scores

In [28]:
# Step 4 — Final AI Search Pipeline

def ai_search(query, top_k=5):

    print("\n==============================")
    print("AI SEARCH PIPELINE STARTED")
    print("==============================")

    # Retrieval
    candidate_indices = retrieve_candidates(
        query
    )

    # Features
    feature_matrix = build_feature_matrix(
        query,
        candidate_indices
    )

    # AI Ranking
    ranked_positions, scores = rank_products(
        feature_matrix
    )

    # Final ranking
    final_indices = [
        candidate_indices[i]
        for i in ranked_positions[:top_k]
    ]

    results = df.iloc[final_indices]

    print("\nFinal AI-ranked products ready!")

    print("\n==============================")
    print("PIPELINE COMPLETED")
    print("==============================")

    return results[
        [
            "title",
            "price",
            "rating"
        ]
    ]

In [29]:
ai_search("कैप")


AI SEARCH PIPELINE STARTED

[Retrieval Pipeline Started]



NameError: name 'bm25' is not defined